In [1]:
import pickle
import pandas as pd
import os

os.chdir('..')
os.chdir('..')
from analytics import RaceData
os.chdir('notebooks/simulation')

In [2]:
rd = RaceData()
rd.add_races_by_date('2026-01-01','2026-03-31',section_results=False)
rddf = rd.results_df.copy()

rddf['Finished'] = (rddf['Running / Reason Out'] == 'Running').astype(int)
rddf['Driver'] = rddf.Driver.str.replace(' (R)','')

In [3]:
with open('iters_0-4999_run2.pickle','rb') as f:
    iters = pickle.load(f)

dfs = []
for i,dfi in enumerate(iters):
    dfi.loc[dfi.Driver == 'Hauger, Denis','Driver'] = 'Hauger, Dennis'
    dfi.loc[dfi.Driver == 'Veekay, Rinus','Driver'] = 'VeeKay, Rinus'
    df = pd.concat([dfi,rddf[['Driver','Pos','SP','Finished','Pts']]]).fillna(0)
    df['Win'] = df.Pos.apply(lambda x: 1 if x==1 else 0)
    df['DNF'] = (1-df.Finished)
    df['Pole'] = df.SP.apply(lambda x: 1 if x==1 else 0)
    df['Indy500Winner'] = ((df.RaceInSeason ==7) & (df.Pos ==1)).astype(int)
    df_agg = df.groupby('Driver').agg(
        Pts = ('Pts','sum'),
        AvgSP = ('SP','mean'),
        FinishPct = ('Finished','mean'),
        AvgPos = ('Pos','mean'),
        Wins = ('Win','sum'),
        Indy500Winner = ('Indy500Winner','max'),
        DNFs = ('DNF','sum'),
        Poles = ('Pole','sum')
    ).reset_index()
    df_agg['Rank'] = df_agg['Pts'].rank(ascending=False)
    df_agg['iter'] = str(i)
    df_agg['Champion'] = (df_agg['Rank'] == 1).astype(int)
    dfs.append(df_agg)

dfa = pd.concat(dfs)    

In [4]:
dfa.groupby('Driver').agg(
        Avg_Pts = ('Pts','mean'),
        FinishRate = ('FinishPct','mean'),
        AvgSP = ('AvgSP','mean'),
        AvgFP = ('AvgPos','mean'),
        AvgWins = ('Wins','mean'),
        AvgPoles = ('Poles','mean'),
        AvgStandings = ('Rank','mean'),
        Championships = ('Champion','sum'),
        Indy500Wins = ('Indy500Winner','sum'),
        Best = ('Rank','min'),
        Worst = ('Rank','max'),
        
).sort_values('AvgStandings')
        

,Avg_Pts,FinishRate,AvgSP,AvgFP,AvgWins,AvgPoles,AvgStandings,Championships,Indy500Wins,Best,Worst
Driver,,,,,,,,,,,
"Palou, Alex",657.373365,0.826056,5.013889,6.922567,5.5606,3.6802,1.2788,4061,1240,1.0,9.0
"O'Ward, Pato",566.295631,0.884844,6.129456,7.529644,1.9688,2.6482,2.7856,628,834,1.0,15.0
"Lundgaard, Christian",509.098744,0.884278,10.514467,8.697867,0.9182,0.8056,4.5758,91,385,1.0,14.0
"Kirkwood, Kyle",508.073269,0.881667,10.292100,9.133789,1.7822,0.4584,4.6174,70,158,1.0,15.0
"Malukas, David",490.364182,0.883756,6.474122,9.838278,1.2346,3.2178,5.4936,66,475,1.0,17.0
"McLaughlin, Scott",485.951872,0.883178,10.452178,10.109344,1.5592,1.8202,5.6978,72,241,1.0,19.0
"Newgarden, Josef",454.107223,0.882422,10.482622,10.883167,1.7362,1.0150,7.2172,8,56,1.0,19.0
"Armstrong, Marcus",430.756729,0.882400,8.440289,10.727844,0.4604,0.6428,8.4362,4,181,1.0,21.0
"Dixon, Scott",422.373497,0.828333,15.552011,11.078178,0.3282,0.0454,8.9154,0,126,2.0,20.0


In [5]:
with open('./iters_0-4999_with_comps_run2.pickle','rb') as f:
    iters_wc = pickle.load(f)

dfs2 = []
for i,dfi in enumerate(iters_wc):
    dfi.loc[dfi.Driver == 'Hauger, Denis','Driver'] = 'Hauger, Dennis'
    dfi.loc[dfi.Driver == 'Veekay, Rinus','Driver'] = 'VeeKay, Rinus'
    df = pd.concat([dfi,rddf[['Driver','Pos','SP','Finished','Pts']]]).fillna(0)
    df['Win'] = df.Pos.apply(lambda x: 1 if x==1 else 0)
    df['DNF'] = (1-df.Finished)
    df['Pole'] = df.SP.apply(lambda x: 1 if x==1 else 0)
    df['Indy500Winner'] = ((df.RaceInSeason ==7) & (df.Pos ==1)).astype(int)
    df_agg = df.groupby('Driver').agg(
        Pts = ('Pts','sum'),
        AvgSP = ('SP','mean'),
        FinishPct = ('Finished','mean'),
        AvgPos = ('Pos','mean'),
        Wins = ('Win','sum'),
        Indy500Winner = ('Indy500Winner','max'),
        DNFs = ('DNF','sum'),
        Poles = ('Pole','sum')
    ).reset_index()
    df_agg['Rank'] = df_agg['Pts'].rank(ascending=False)
    df_agg['iter'] = f'{i}_wc'
    df_agg['Champion'] = (df_agg['Rank'] == 1).astype(int)
    dfs2.append(df_agg)

dfa2 = pd.concat(dfs2)    

In [6]:
dfa2.groupby('Driver').agg(
        Avg_Pts = ('Pts','mean'),
        FinishRate = ('FinishPct','mean'),
        AvgSP = ('AvgSP','mean'),
        AvgFP = ('AvgPos','mean'),
        AvgWins = ('Wins','mean'),
        AvgPoles = ('Poles','mean'),
        AvgStandings = ('Rank','mean'),
        Championships = ('Champion','sum'),
        Indy500Wins = ('Indy500Winner','sum'),
        Best = ('Rank','min'),
        Worst = ('Rank','max')
).sort_values('AvgStandings')
        

,Avg_Pts,FinishRate,AvgSP,AvgFP,AvgWins,AvgPoles,AvgStandings,Championships,Indy500Wins,Best,Worst
Driver,,,,,,,,,,,
"Palou, Alex",655.838009,0.828133,5.042111,6.954744,5.5118,3.6922,1.2698,4091,1207,1.0,7.0
"O'Ward, Pato",563.881146,0.883456,6.171878,7.607278,1.9430,2.6524,2.7908,592,829,1.0,13.0
"Lundgaard, Christian",504.627413,0.882389,10.582867,8.853267,0.9144,0.7940,4.6510,69,379,1.0,16.0
"Kirkwood, Kyle",504.342147,0.882800,10.272900,9.281578,1.7838,0.4596,4.6608,69,187,1.0,14.0
"Malukas, David",489.414632,0.883900,6.529178,9.895011,1.2436,3.1886,5.4236,91,436,1.0,18.0
"McLaughlin, Scott",484.298920,0.885144,10.498567,10.167011,1.5556,1.8130,5.6396,71,250,1.0,19.0
"Newgarden, Josef",452.300748,0.881733,10.464844,10.970700,1.7732,1.0092,7.1052,12,66,1.0,20.0
"Armstrong, Marcus",428.736455,0.883267,8.486500,10.821833,0.4764,0.6504,8.3654,3,190,1.0,21.0
"Dixon, Scott",418.805374,0.827622,15.517433,11.216600,0.3386,0.0462,8.9420,0,86,2.0,21.0


In [7]:
df_all = pd.concat([dfa,dfa2])
df_all['FinishPct2'] = df_all['FinishPct']*100
print(df_all.groupby('Driver').agg(
        Avg_Pts = ('Pts','mean'),
        FinishRate = ('FinishPct2','mean'),
        AvgSP = ('AvgSP','mean'),
        AvgFP = ('AvgPos','mean'),
        AvgWins = ('Wins','mean'),
        AvgPoles = ('Poles','mean'),
).sort_values('Avg_Pts',ascending=False).round(1).to_markdown())
print(df_all.groupby('Driver').agg(
        AvgStandings = ('Rank','mean'),
        Championships = ('Champion','sum'),
        Indy500Wins = ('Indy500Winner','sum'),
        Best = ('Rank','min'),
        Worst = ('Rank','max')
).sort_values('AvgStandings').round(1).to_markdown())

| Driver               |   Avg_Pts |   FinishRate |   AvgSP |   AvgFP |   AvgWins |   AvgPoles |
|:---------------------|----------:|-------------:|--------:|--------:|----------:|-----------:|
| Palou, Alex          |     656.6 |         82.7 |     5   |     6.9 |       5.5 |        3.7 |
| O'Ward, Pato         |     565.1 |         88.4 |     6.2 |     7.6 |       2   |        2.7 |
| Lundgaard, Christian |     506.9 |         88.3 |    10.5 |     8.8 |       0.9 |        0.8 |
| Kirkwood, Kyle       |     506.2 |         88.2 |    10.3 |     9.2 |       1.8 |        0.5 |
| Malukas, David       |     489.9 |         88.4 |     6.5 |     9.9 |       1.2 |        3.2 |
| McLaughlin, Scott    |     485.1 |         88.4 |    10.5 |    10.1 |       1.6 |        1.8 |
| Newgarden, Josef     |     453.2 |         88.2 |    10.5 |    10.9 |       1.8 |        1   |
| Armstrong, Marcus    |     429.7 |         88.3 |     8.5 |    10.8 |       0.5 |        0.6 |
| Dixon, Scott         |     4

In [19]:
dfdiff = pd.read_csv('pred differences.csv')
print(dfdiff[['**Driver**', 'Average Season Points', 'Average Finishing Position',
       'Wins per Season', 'Average Championship Standing',
       '  Total Championships', ' Total Indy 500 Wins']].sort_values('Average Season Points',ascending=False).to_markdown(index=False))

| **Driver**               |   Average Season Points |   Average Finishing Position |   Wins per Season |   Average Championship Standing |     Total Championships |    Total Indy 500 Wins |
|:-------------------------|------------------------:|-----------------------------:|------------------:|--------------------------------:|------------------------:|-----------------------:|
| **Kirkwood, Kyle**       |                    65.1 |                         -1.9 |               0.9 |                            -3.5 |                     117 |                    104 |
| **Ericsson, Marcus**     |                    40.2 |                         -1.7 |               0   |                            -4.6 |                       0 |                     42 |
| **Malukas, David**       |                    39.5 |                         -1.5 |               0.1 |                            -2.2 |                     109 |                    313 |
| **Hauger, Dennis**       |                 